<a id="top"></a>
# BB84 detector study

**Question.** At a matched 5% false-alarm target, when do basis- and state-conditioned scores detect partial intercept-resend better than pooled QBER, and how sensitive is that comparison to finite baseline fitting and three interim looks?

The Monte Carlo detector comparison runs locally with NumPy and SciPy.

## Run

Install the repository requirements, edit the configuration cell below, and run top to bottom. Use `is_attack` to identify null rows, where `eve_p_z = 0.5` is a placeholder.

The terminal sweep produces `bb84_sweep_results` and `bb84_calibration_summary`. The cumulative sweep produces `bb84_cumulative_results`.

## Navigate

- [Configuration](#configuration)
- [Method and code map](#method-map)
- [Detector setup](#setup)
- [Terminal detector sweeps](#terminal-sweeps)
- [Terminal results](#terminal-results)
- [Calibration-fit summary](#calibration-summary)
- [Cumulative alarms](#cumulative-alarms)
- [References](#references)


<a id="configuration"></a>
## Configuration


In [ ]:
# Enable either or both BB84 detector comparisons.
RUN_TERMINAL_SWEEP = True  # One decision at the end of each trial.
RUN_CUMULATIVE_SWEEP = True  # Any alarm across the scheduled checks.

# Settings shared by both sweeps.
TARGET_FALSE_ALARM = 0.05  # Target false-alarm probability per trial.
CHECK_BUDGETS = (20, 50, 100)  # Number of retained checks per trial.
EVE_Z_PROBABILITIES = (0.1, 0.5, 0.9)  # Probability that Eve measures in Z.
# Fraction of transmissions Eve intercepts. The sweeps add a zero-interception control.
SWEEP_INTERCEPT_FRACTIONS = (0.005, 0.01, 0.025, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75, 1.0)
SWEEP_CALIBRATION_SIZES = (1_000, 10_000)  # Baseline calibration trials per state.
SWEEP_BASELINE_FITS = 4  # Independent rate estimates per calibration size.

# Noise profiles are (label, Z error rate, X error rate).
SWEEP_NOISE_PROFILES = [
    ("No noise", 0.0, 0.0),
    ("Erven reference", 0.002, 0.038),
]
# Add balanced and basis-biased profiles at each synthetic mean error rate.
for mean in (0.01, 0.02, 0.04):
    low, high = mean * (1 - 0.9), mean * (1 + 0.9)
    SWEEP_NOISE_PROFILES.extend([
        (f"{mean:.0%} mean, balanced", mean, mean),
        (f"{mean:.0%} mean, higher X", low, high),
        (f"{mean:.0%} mean, higher Z", high, low),
    ])

# Terminal sweep seed and sample sizes for cutoff calibration and evaluation.
SWEEP_SEED = 20261031
SWEEP_CUTOFF_TRIALS = 5_000
SWEEP_EVALUATION_TRIALS = 5_000

# Cumulative sweep seed and sample sizes. Looks must be increasing fractions
# ending at 1.0 and round up to distinct check counts for each budget.
LOOK_FRACTIONS = (0.25, 0.5, 1.0)
CUMULATIVE_SEED = 20261101
CUMULATIVE_CUTOFF_TRIALS = 3_000
CUMULATIVE_EVALUATION_TRIALS = 3_000

# Choose a plotted scenario from the grid above. Use size 0 and fit 1
# to plot the known-baseline result.
VIEW_NOISE_PROFILE = "2% mean, higher X"
VIEW_CALIBRATION_SIZE = 1_000
VIEW_BASELINE_FIT = 1
VIEW_EVE_P_Z = 0.5


<a id="method-map"></a>
## Method and code map

- `sample_bb84_checks` models preparation, sifting, and partial intercept-resend [R1–R3]. Uniform Eve basis choice gives attack-only sifted QBER `f/4`. The project adds biased-Eve XOR errors.
- `detector_scores` compares pooled QBER, basis `U2`, and state `U4`. Basis-resolved estimation follows [R4]. The project defines the score weights.
- `calibrate_bb84_cutoffs` and `run_bb84_cumulative_sweep` use independent null samples and familywise calibration across looks, motivated by finite-statistical control [R6].
- `detector_normal_moments` follows the normal-tail comparison in [R3]. `wilson_95_interval` reports Monte Carlo uncertainty [R7].

The Erven reference uses `(q_Z, q_X) = (0.002, 0.038)` [R5, §4]. The other profiles are synthetic sensitivity settings.


<a id="setup"></a>
## 1. Detector setup

The code below defines how the three detectors score a set of retained BB84 checks.


### Detector definitions and controls

| Detector | Statistic | Interpretation |
| --- | --- | --- |
| Pooled QBER | Total errors divided by total retained checks | Ignores which basis or prepared state produced an error. |
| Basis `U2` | One-sided score after grouping the two states in each basis | Can use a difference between fitted Z- and X-basis error rates. |
| State `U4` | One-sided score using all four fitted state error rates | Can use within-basis state differences; it intentionally equals `U2` when paired fitted rates are exactly equal. |

All detectors see the same checks. Fitting, cutoff calibration, null evaluation, and attack evaluation use independent seeded streams. Verify held-out null rates before comparing power.


In [ ]:
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm

STATE_ORDER = ("Z0", "Z1", "X+", "X-")
DETECTOR_NAMES = ("Pooled QBER", "Basis U2", "State U4")


def sample_bb84_checks(
    rng, intercept_fraction, eve_p_z, *,
    retained_checks, trials, state_baseline,
):
    """Sample state totals and errors for fixed-size sifted BB84 trials."""
    counts = rng.multinomial(retained_checks, [0.25] * len(STATE_ORDER), size=trials)
    attack_error = 0.5 * intercept_fraction * np.repeat([1 - eve_p_z, eve_p_z], 2)
    error_rate = state_baseline + (1 - 2 * state_baseline) * attack_error
    return counts, rng.binomial(counts, error_rate)


def conditioned_score(counts, errors, baseline):
    variable = (baseline > 0) & (baseline < 1)
    rates = baseline[variable]
    weights = (1 - 2 * rates) / (rates * (1 - rates))
    score = ((errors[:, variable] - counts[:, variable] * rates) * weights).sum(axis=1)

    # A true rate of 0 or 1 makes any deviation conclusive evidence against the null.
    expected = counts[:, ~variable] * baseline[~variable]
    score[(errors[:, ~variable] != expected).any(axis=1)] = np.inf
    return score


def detector_scores(counts, errors, fitted_baseline):
    """Return pooled QBER, basis-grouped U2, and state-conditioned U4."""
    basis_baseline = fitted_baseline.reshape(2, 2).mean(axis=1)
    basis_counts = counts.reshape(-1, 2, 2).sum(axis=2)
    basis_errors = errors.reshape(-1, 2, 2).sum(axis=2)
    basis_score = conditioned_score(basis_counts, basis_errors, basis_baseline)
    if np.array_equal(fitted_baseline[::2], fitted_baseline[1::2]):
        # With equal rates within each basis, the two scores are algebraically identical.
        state_score = basis_score
    else:
        state_score = conditioned_score(counts, errors, fitted_baseline)
    return np.column_stack([errors.sum(axis=1) / counts.sum(axis=1), basis_score, state_score])

<a id="terminal-sweeps"></a>
## 2. Terminal detector sweeps

Sweep retained checks, Eve's interception and basis bias, baseline noise, and calibration size. Each trial has one terminal decision.

For each fitted baseline and budget, an independent null batch sets the cutoff, including randomized ties at the configured false-alarm target. Held-out paired batches estimate false alarms and power.

Noise rates are equal within each basis, so true-rate `U2` and `U4` coincide. Finite fits can separate them. Wilson intervals apply to one fit. Repeated fits show calibration sensitivity.

Dashed curves use unconditional normal moments. Analytic conditioned scores are omitted at singular endpoint baselines.

Output: `bb84_sweep_results`, one row per profile, budget, fit, attack setting, and detector.


In [ ]:
def calibrate_bb84_cutoffs(null_scores, target):
    """Return each upper-tail cutoff and its probability of alarming on a tie."""
    rank = int(np.ceil((1 - target) * len(null_scores))) - 1
    cutoffs = np.partition(null_scores, rank, axis=0)[rank]
    above = (null_scores > cutoffs).mean(axis=0)
    tied = (null_scores == cutoffs).mean(axis=0)
    return cutoffs, np.clip((target - above) / tied, 0.0, 1.0)


def wilson_95_interval(events, trials):
    rate = events / trials
    z = 1.959963984540054
    denominator = 1 + z**2 / trials
    center = (rate + z**2 / (2 * trials)) / denominator
    radius = z * np.sqrt(rate * (1 - rate) / trials + z**2 / (4 * trials**2)) / denominator
    return np.clip(center - radius, 0.0, 1.0), np.clip(center + radius, 0.0, 1.0)


def detector_normal_moments(retained_checks, true_baseline, intercept_fraction, eve_p_z, fitted_baseline):
    """Normal moments under multinomially sampled prepared states."""
    attack_error = 0.5 * intercept_fraction * np.repeat([1 - eve_p_z, eve_p_z], 2)
    alternative = true_baseline + (1 - 2 * true_baseline) * attack_error
    pooled_mean = alternative.mean()
    means = [pooled_mean]
    variances = [pooled_mean * (1 - pooled_mean) / retained_checks]

    basis_fit = fitted_baseline.reshape(2, 2).mean(axis=1).repeat(2)
    for detector_fit in (basis_fit, fitted_baseline):
        if np.any((detector_fit <= 0) | (detector_fit >= 1)):
            means.append(np.nan)
            variances.append(np.nan)
            continue
        weights = (1 - 2 * detector_fit) / (detector_fit * (1 - detector_fit))
        state_means = weights * (alternative - detector_fit)
        per_bit_mean = state_means.mean()
        per_bit_variance = np.mean(
            weights**2 * alternative * (1 - alternative)
            + (state_means - per_bit_mean) ** 2
        )
        means.append(retained_checks * per_bit_mean)
        variances.append(retained_checks * per_bit_variance)
    return np.asarray(means), np.sqrt(np.asarray(variances))


def normal_alarm_probability(
    cutoffs, tie_rates, means, standard_deviations, retained_checks
):
    probabilities = np.empty(len(cutoffs))
    for i, (cutoff, tie_rate, mean, deviation) in enumerate(
        zip(cutoffs, tie_rates, means, standard_deviations)
    ):
        if np.isnan(deviation):
            probabilities[i] = np.nan
        elif deviation == 0:
            probabilities[i] = float(mean > cutoff) + tie_rate * float(mean == cutoff)
        elif i == 0:
            cutoff_count = cutoff * retained_checks
            mean_count = mean * retained_checks
            deviation_count = deviation * retained_checks
            above = norm.sf(
                (cutoff_count + 0.5 - mean_count) / deviation_count
            )
            tied = norm.cdf(
                (cutoff_count + 0.5 - mean_count) / deviation_count
            ) - norm.cdf(
                (cutoff_count - 0.5 - mean_count) / deviation_count
            )
            probabilities[i] = above + tie_rate * tied
        else:
            probabilities[i] = norm.sf((cutoff - mean) / deviation)
    return probabilities

In [ ]:
def run_bb84_sweep():
    fit_rng, cutoff_rng, null_rng, attack_rng = [
        np.random.default_rng(seed) for seed in np.random.SeedSequence(SWEEP_SEED).spawn(4)
    ]
    # Use a neutral Eve-basis reference for attack-free rows to keep the column numeric.
    scenarios = [(0.0, 0.5, False)]
    scenarios.extend((fraction, eve_p_z, True) for fraction, eve_p_z in product(
        SWEEP_INTERCEPT_FRACTIONS, EVE_Z_PROBABILITIES
    ))
    rows = []
    for noise, q_z, q_x in SWEEP_NOISE_PROFILES:
        true_rates = np.repeat([q_z, q_x], 2)
        fits = [(0, 1, true_rates)]
        for size in SWEEP_CALIBRATION_SIZES:
            errors = fit_rng.binomial(size, true_rates, size=(SWEEP_BASELINE_FITS, len(STATE_ORDER)))
            rates = (errors + 0.5) / (size + 1)
            fits.extend((size, index, fit) for index, fit in enumerate(rates, start=1))

        for budget in CHECK_BUDGETS:
            sampling = dict(retained_checks=budget, state_baseline=true_rates)
            cutoff_checks = sample_bb84_checks(
                cutoff_rng, 0.0, 0.5, trials=SWEEP_CUTOFF_TRIALS, **sampling
            )
            rules = [
                calibrate_bb84_cutoffs(detector_scores(*cutoff_checks, fit), TARGET_FALSE_ALARM)
                for _, _, fit in fits
            ]

            for fraction, eve_p_z, is_attack in scenarios:
                rng = attack_rng if is_attack else null_rng
                checks = sample_bb84_checks(
                    rng, fraction, eve_p_z,
                    trials=SWEEP_EVALUATION_TRIALS, **sampling
                )
                tie_draws = rng.random((SWEEP_EVALUATION_TRIALS, 1))
                for (size, fit_id, fit), (cutoffs, tie_rates) in zip(fits, rules):
                    scores = detector_scores(*checks, fit)
                    alarms = (scores > cutoffs) | ((scores == cutoffs) & (tie_draws < tie_rates))
                    event_counts = alarms.sum(axis=0)
                    lower, upper = wilson_95_interval(event_counts, SWEEP_EVALUATION_TRIALS)
                    normal_means, normal_deviations = detector_normal_moments(
                        budget, true_rates, fraction, eve_p_z, fit
                    )
                    normal_rates = normal_alarm_probability(
                        cutoffs, tie_rates, normal_means, normal_deviations, budget
                    )

                    for i, detector in enumerate(DETECTOR_NAMES):
                        rows.append({
                            "noise_profile": noise,
                            "q_z": q_z,
                            "q_x": q_x,
                            "retained_checks": budget,
                            "calibration_trials_per_state": size,
                            "baseline_fit": fit_id,
                            "qhat_z0": fit[0],
                            "qhat_z1": fit[1],
                            "qhat_x_plus": fit[2],
                            "qhat_x_minus": fit[3],
                            "intercept_fraction": fraction,
                            "eve_p_z": eve_p_z,
                            "is_attack": is_attack,
                            "detector": detector,
                            "alarm_count": event_counts[i],
                            "trials": SWEEP_EVALUATION_TRIALS,
                            "alarm_rate": event_counts[i] / SWEEP_EVALUATION_TRIALS,
                            "normal_approximation": normal_rates[i],
                            "lower_95": lower[i],
                            "upper_95": upper[i],
                            "cutoff": cutoffs[i],
                            "cutoff_tie_probability": tie_rates[i],
                        })
    return pd.DataFrame(rows)


bb84_sweep_results = run_bb84_sweep() if RUN_TERMINAL_SWEEP else None
if bb84_sweep_results is not None:
    display(bb84_sweep_results.head())

<a id="terminal-results"></a>
### Interpret terminal results

Null rows (`is_attack = False`) give held-out false alarms. Attack rows give power. Select a profile, calibration size, fit, and Eve bias in the configuration cell at the top.

Solid curves are Monte Carlo rates, dashed curves are normal approximations, and bands are 95% Wilson intervals. The 80% column is the first tested interception fraction reaching that power, without interpolation.


In [ ]:
if RUN_TERMINAL_SWEEP:
    view = bb84_sweep_results[
        bb84_sweep_results["noise_profile"].eq(VIEW_NOISE_PROFILE)
        & bb84_sweep_results["calibration_trials_per_state"].eq(VIEW_CALIBRATION_SIZE)
        & bb84_sweep_results["baseline_fit"].eq(VIEW_BASELINE_FIT)
    ]
    if view.empty or VIEW_EVE_P_Z not in EVE_Z_PROBABILITIES:
        raise ValueError("Select a noise profile, calibration size, fit, and Eve basis from the sweep.")

    power = view[view["is_attack"] & view["eve_p_z"].eq(VIEW_EVE_P_Z)]
    fig, axes = plt.subplots(
        1, len(CHECK_BUDGETS), figsize=(4.6 * len(CHECK_BUDGETS), 4.2),
        sharey=True, squeeze=False, layout="constrained"
    )
    for ax, budget in zip(axes[0], CHECK_BUDGETS):
        budget_view = view[view["retained_checks"].eq(budget)]
        for detector in DETECTOR_NAMES:
            curve = budget_view[
                budget_view["detector"].eq(detector)
                & (~budget_view["is_attack"] | budget_view["eve_p_z"].eq(VIEW_EVE_P_Z))
            ].sort_values("intercept_fraction")
            x = curve["intercept_fraction"].to_numpy()
            line, = ax.plot(x, curve["alarm_rate"].to_numpy(), marker="o", ms=3, label=detector)
            ax.plot(
                x, curve["normal_approximation"].to_numpy(),
                color=line.get_color(), linestyle="--", linewidth=1
            )
            ax.fill_between(
                x, curve["lower_95"].to_numpy(), curve["upper_95"].to_numpy(),
                color=line.get_color(), alpha=0.15
            )
        ax.axhline(0.8, color="0.5", linestyle=":", linewidth=1)
        ax.set(title=f"{budget} retained checks", xlabel="Interception fraction", ylim=(0, 1.02))
    axes[0, 0].set_ylabel("Alarm probability")
    axes[0, -1].legend(loc="lower right")
    fig.suptitle(
        f"{VIEW_NOISE_PROFILE} · Eve Z probability {VIEW_EVE_P_Z:g}"
        " · solid Monte Carlo, dashed normal approximation"
    )
    plt.show()

    keys = ["retained_checks", "detector"]
    false_alarms = view[~view["is_attack"]].set_index(keys)[
        ["alarm_rate", "lower_95", "upper_95"]
    ].rename(columns={"alarm_rate": "false_alarm_rate"})
    first_at_80 = (
        power[power["alarm_rate"].ge(0.8)].groupby(keys)["intercept_fraction"].min()
        .rename("first_tested_fraction_at_80pct_power")
    )
    display(false_alarms.join(first_at_80).round(4))

<a id="calibration-summary"></a>
## 3. Calibration-fit summary

`bb84_calibration_summary` shows how alarm rates vary across four independent fitted baselines.


In [ ]:
bb84_calibration_summary = None
if RUN_TERMINAL_SWEEP:
    bb84_calibration_summary = (
        bb84_sweep_results[
            bb84_sweep_results["calibration_trials_per_state"].gt(0)
        ]
        .groupby([
            "noise_profile", "retained_checks",
            "calibration_trials_per_state", "intercept_fraction",
            "eve_p_z", "is_attack", "detector",
        ], as_index=False)
        .agg(
            mean_alarm_rate=("alarm_rate", "mean"),
            between_fit_std=("alarm_rate", "std"),
            minimum_alarm_rate=("alarm_rate", "min"),
            maximum_alarm_rate=("alarm_rate", "max"),
            mean_normal_approximation=("normal_approximation", "mean"),
        )
    )
    display(bb84_calibration_summary.head())


<a id="cumulative-alarms"></a>
## 4. Cumulative alarms

By default, each detector evaluates at 25%, 50%, and 100% of the retained-check budget. A separate null batch calibrates the maximum standardized score to the configured probability of any alarm across looks.

`bb84_cumulative_results` contains any-alarm rates, Wilson intervals, first-alarm timing, and reach probabilities. Check null `any_alarm_rate` before comparing attack power. `mean_first_alarm_checks` is conditional on an alarm.

The schedule comes from the proposal. Maximum-over-looks calibration follows the project's finite-statistical motivation [R6].


In [ ]:
def bb84_look_counts(retained_checks):
    return np.unique(np.maximum(
        1, np.ceil(retained_checks * np.asarray(LOOK_FRACTIONS)).astype(int)
    ))


def sample_bb84_check_looks(
    rng, looks, intercept_fraction, eve_p_z, *, trials, state_baseline
):
    counts = np.zeros((trials, len(STATE_ORDER)), dtype=int)
    errors = np.zeros_like(counts)
    counts_at_looks, errors_at_looks = [], []
    previous = 0
    for look in looks:
        increment = sample_bb84_checks(
            rng, intercept_fraction, eve_p_z,
            retained_checks=int(look - previous), trials=trials,
            state_baseline=state_baseline,
        )
        counts += increment[0]
        errors += increment[1]
        counts_at_looks.append(counts.copy())
        errors_at_looks.append(errors.copy())
        previous = look
    return np.stack(counts_at_looks, axis=1), np.stack(errors_at_looks, axis=1)


def detector_scores_at_looks(counts, errors, fitted_baseline):
    return np.stack([
        detector_scores(counts[:, look], errors[:, look], fitted_baseline)
        for look in range(counts.shape[1])
    ], axis=1)


def cumulative_null_standardization(looks, fitted_baseline):
    centers, scales = [], []
    for look in looks:
        means, deviations = detector_normal_moments(
            int(look), fitted_baseline, 0.0, 0.5, fitted_baseline
        )
        centers.append([means[0], 0.0, 0.0])
        scales.append(np.nan_to_num(deviations, nan=0.0))
    return np.asarray(centers), np.asarray(scales)


def standardize_look_scores(scores, centers, scales):
    delta = scores - centers
    standardized = np.zeros_like(delta, dtype=float)
    np.divide(delta, scales, out=standardized, where=scales > 0)
    zero_scale = np.broadcast_to(scales == 0, delta.shape)
    standardized[zero_scale & (delta > 0)] = np.inf
    standardized[zero_scale & (delta < 0)] = -np.inf
    return standardized


def run_bb84_cumulative_sweep():
    fit_rng, cutoff_rng, null_rng, attack_rng = [
        np.random.default_rng(seed)
        for seed in np.random.SeedSequence(CUMULATIVE_SEED).spawn(4)
    ]
    scenarios = [(0.0, 0.5, False)]
    scenarios.extend((fraction, eve_p_z, True) for fraction, eve_p_z in product(
        SWEEP_INTERCEPT_FRACTIONS, EVE_Z_PROBABILITIES
    ))
    rows = []
    for noise, q_z, q_x in SWEEP_NOISE_PROFILES:
        true_rates = np.repeat([q_z, q_x], 2)
        fits = [(0, 1, true_rates)]
        for size in SWEEP_CALIBRATION_SIZES:
            calibration_errors = fit_rng.binomial(
                size, true_rates, size=(SWEEP_BASELINE_FITS, len(STATE_ORDER))
            )
            fitted_rates = (calibration_errors + 0.5) / (size + 1)
            fits.extend(
                (size, index, fit)
                for index, fit in enumerate(fitted_rates, start=1)
            )

        for budget in CHECK_BUDGETS:
            looks = bb84_look_counts(budget)
            cutoff_counts, cutoff_errors = sample_bb84_check_looks(
                cutoff_rng, looks, 0.0, 0.5,
                trials=CUMULATIVE_CUTOFF_TRIALS, state_baseline=true_rates,
            )
            rules = []
            for _, _, fit in fits:
                centers, scales = cumulative_null_standardization(looks, fit)
                cutoff_scores = detector_scores_at_looks(
                    cutoff_counts, cutoff_errors, fit
                )
                maxima = standardize_look_scores(
                    cutoff_scores, centers, scales
                ).max(axis=1)
                cutoffs, tie_rates = calibrate_bb84_cutoffs(
                    maxima, TARGET_FALSE_ALARM
                )
                rules.append((centers, scales, cutoffs, tie_rates))

            for fraction, eve_p_z, is_attack in scenarios:
                rng = attack_rng if is_attack else null_rng
                counts, errors = sample_bb84_check_looks(
                    rng, looks, fraction, eve_p_z,
                    trials=CUMULATIVE_EVALUATION_TRIALS,
                    state_baseline=true_rates,
                )
                tie_draws = rng.random((CUMULATIVE_EVALUATION_TRIALS, 1, 1))
                for (size, fit_id, fit), rule in zip(fits, rules):
                    centers, scales, cutoffs, tie_rates = rule
                    scores = detector_scores_at_looks(counts, errors, fit)
                    standardized = standardize_look_scores(
                        scores, centers, scales
                    )
                    alarm_at_look = (
                        (standardized > cutoffs)
                        | (
                            (standardized == cutoffs)
                            & (tie_draws < tie_rates.reshape(1, 1, -1))
                        )
                    )
                    any_alarm = alarm_at_look.any(axis=1)
                    event_counts = any_alarm.sum(axis=0)
                    lower, upper = wilson_95_interval(
                        event_counts, CUMULATIVE_EVALUATION_TRIALS
                    )
                    for detector_index, detector in enumerate(DETECTOR_NAMES):
                        detector_alarms = alarm_at_look[:, :, detector_index]
                        ever = any_alarm[:, detector_index]
                        first_indices = np.argmax(detector_alarms, axis=1)
                        first_checks = looks[first_indices]
                        row = {
                            "noise_profile": noise,
                            "q_z": q_z,
                            "q_x": q_x,
                            "retained_checks": budget,
                            "look_counts": tuple(int(value) for value in looks),
                            "calibration_trials_per_state": size,
                            "baseline_fit": fit_id,
                            "intercept_fraction": fraction,
                            "eve_p_z": eve_p_z,
                            "is_attack": is_attack,
                            "detector": detector,
                            "alarm_count": event_counts[detector_index],
                            "trials": CUMULATIVE_EVALUATION_TRIALS,
                            "any_alarm_rate": (
                                event_counts[detector_index]
                                / CUMULATIVE_EVALUATION_TRIALS
                            ),
                            "lower_95": lower[detector_index],
                            "upper_95": upper[detector_index],
                            "maximum_cutoff": cutoffs[detector_index],
                            "cutoff_tie_probability": tie_rates[detector_index],
                            "mean_first_alarm_checks": (
                                first_checks[ever].mean()
                                if ever.any() else np.nan
                            ),
                        }
                        for look_index, look in enumerate(looks):
                            label = int(round(
                                100 * LOOK_FRACTIONS[look_index]
                            ))
                            row[f"alarm_by_{label}pct"] = detector_alarms[
                                :, :look_index + 1
                            ].any(axis=1).mean()
                            row[f"first_alarm_at_{label}pct"] = (
                                ever & (first_indices == look_index)
                            ).mean()
                            row[f"reach_{label}pct"] = (
                                ~detector_alarms[
                                    :, :look_index
                                ].any(axis=1)
                            ).mean()
                        rows.append(row)
    return pd.DataFrame(rows)


bb84_cumulative_results = run_bb84_cumulative_sweep() if RUN_CUMULATIVE_SWEEP else None
if bb84_cumulative_results is not None:
    display(bb84_cumulative_results.head())


In [ ]:
if RUN_CUMULATIVE_SWEEP:
    cumulative_view = bb84_cumulative_results[
        bb84_cumulative_results["noise_profile"].eq(VIEW_NOISE_PROFILE)
        & bb84_cumulative_results[
            "calibration_trials_per_state"
        ].eq(VIEW_CALIBRATION_SIZE)
        & bb84_cumulative_results["baseline_fit"].eq(VIEW_BASELINE_FIT)
    ]
    if cumulative_view.empty or VIEW_EVE_P_Z not in EVE_Z_PROBABILITIES:
        raise ValueError("Select a noise profile, calibration size, fit, and Eve basis from the sweep.")

    cumulative_null = cumulative_view[~cumulative_view["is_attack"]]
    cumulative_power = cumulative_view[
        cumulative_view["is_attack"]
        & cumulative_view["eve_p_z"].eq(VIEW_EVE_P_Z)
    ]

    fig, axes = plt.subplots(
        1, len(CHECK_BUDGETS),
        figsize=(4.6 * len(CHECK_BUDGETS), 4.2),
        sharey=True, squeeze=False, layout="constrained",
    )
    for ax, budget in zip(axes[0], CHECK_BUDGETS):
        for detector in DETECTOR_NAMES:
            null_row = cumulative_null[
                cumulative_null["retained_checks"].eq(budget)
                & cumulative_null["detector"].eq(detector)
            ]
            curve = cumulative_power[
                cumulative_power["retained_checks"].eq(budget)
                & cumulative_power["detector"].eq(detector)
            ].sort_values("intercept_fraction")
            x = np.r_[0.0, curve["intercept_fraction"].to_numpy()]
            y = np.r_[
                null_row["any_alarm_rate"].iloc[0],
                curve["any_alarm_rate"].to_numpy(),
            ]
            ax.plot(x, y, marker="o", ms=3, label=detector)
        ax.axhline(0.8, color="0.5", linestyle=":", linewidth=1)
        ax.set(
            title=f"{budget} retained checks",
            xlabel="Interception fraction", ylim=(0, 1.02),
        )
    axes[0, 0].set_ylabel("Probability of any alarm")
    axes[0, -1].legend(loc="lower right")
    fig.suptitle(
        f"Cumulative alarm test · {VIEW_NOISE_PROFILE}"
        f" · Eve Z probability {VIEW_EVE_P_Z:g}"
    )
    plt.show()

    display(cumulative_null.set_index(["retained_checks", "detector"])[
        ["look_counts", "any_alarm_rate", "lower_95", "upper_95"]
    ].round(4))


<a id="references"></a>
## References

- **[R1]** C. H. Bennett and G. Brassard, “Quantum Cryptography: Public Key
  Distribution and Coin Tossing,” *Proceedings of IEEE International Conference
  on Computers, Systems and Signal Processing*, pp. 175–179 (1984).
  [Reprint: arXiv:2003.06557](https://arxiv.org/abs/2003.06557).
- **[R2]** V. Scarani et al., “The Security of Practical Quantum Key
  Distribution,” *Reviews of Modern Physics* **81**, 1301–1350 (2009),
  especially §I.B.5, p. 1305.
  [arXiv:0802.4155](https://arxiv.org/abs/0802.4155).
- **[R3]** Chankyun Lee, Eunjoo Lee, and Wonhyuk Lee, “Analysis on Eavesdropper Detection in
  BB84 Quantum Key Distribution Protocol Against Partial Intercept-and-Resend
  Attack,” *ONDM 2023*, especially §III.A, Eqs. (1)–(12).
  [Open paper](https://dl.ifip.org/db/conf/ondm/ondm2023/1570879964.pdf).
- **[R4]** H.-K. Lo, H. F. Chau, and M. Ardehali, “Efficient Quantum Key
  Distribution Scheme and a Proof of Its Unconditional Security,” *Journal of
  Cryptology* **18**, 133–165 (2005), especially §4.2, pp. 26–28 of the preprint.
  [arXiv:quant-ph/0011056](https://arxiv.org/abs/quant-ph/0011056).
- **[R5]** C. Erven, X. Ma, R. Laflamme, and G. Weihs, “Entangled Quantum Key
  Distribution with a Biased Basis Choice,” *New Journal of Physics* **11**,
  045025 (2009), especially §4, p. 9.
  [arXiv:0901.0960](https://arxiv.org/abs/0901.0960).
- **[R6]** Vaisakh Mannalath, Víctor Zapatero, and Marcos Curty, “Sharp Finite Statistics for
  Quantum Key Distribution,” *Physical Review Letters* **135**, 020803 (2025),
  especially Propositions 1–3.
  [arXiv:2410.04095](https://arxiv.org/abs/2410.04095).
- **[R7]** E. B. Wilson, “Probable Inference, the Law of Succession, and
  Statistical Inference,” *Journal of the American Statistical Association*
  **22**(158), 209–212 (1927).
  [doi:10.1080/01621459.1927.10502953](https://doi.org/10.1080/01621459.1927.10502953).

[Back to top](#top)
